# Block 1 — Deep Learning with Time Series Data: Preprocessing Fundamentals

**Goals for this block:**
- Load and analyze time series datasets
- Implement effective strategies for handling missing values
- Normalize and standardize features for optimal model performance
- Create proper time-aware train/validation/test splits
- Generate sliding windows for sequence modeling

## 0. Setup & Environment

Let's import the necessary libraries for time series analysis and visualization.

In [ ]:
# GUIDED: imports and environment checks
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading and Initial Exploration

We'll work with an **energy generation** dataset that contains typical patterns and challenges found in real-world time series data.

In [ ]:
# GUIDED: load data
df = pd.read_parquet("energy_synthetic.parquet")
# parse a timestamp column smartly
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
df['timestamp'] = df['timestamp'].dt.tz_localize(None)
df = df.set_index('timestamp').sort_index()
df.head(6)

### 1.1 Data Quality Checks

Before preprocessing, we need to verify several important properties:
- **Index integrity**: Is the index properly formatted as a DatetimeIndex and monotonically increasing?
- **Frequency detection**: Can we detect the native sampling frequency of the data?
- **Data completeness**: Are there missing values or duplicate timestamps that need handling?

In [ ]:
# GUIDED: basic info
print(df.info())

# Check duplicates
dups = df.index.duplicated().sum()
print(f'Duplicate timestamps: {dups}')

# infer frequency
inferred = pd.infer_freq(df.index)
print('Inferred frequency:', inferred)

# Check missing
missing = df.isna().sum()
print('Missing values per column:\n', missing)


In [ ]:
# GUIDED: visualize raw series
plt.figure(figsize=(10, 3))
df.iloc[:1000, 0].plot(title='first ~1000 points', xlabel='Timestamp (hour)', ylabel='Value (MW)')
plt.tight_layout()
plt.show()

## 2. Handling Missing Values in Time Series

Missing values in time series require special attention as they can affect temporal patterns. The appropriate strategy depends on your domain knowledge and the characteristics of the data:

- **Forward-fill**: Propagates the last valid observation forward (assumes persistence)
- **Backward-fill**: Uses next known values to fill gaps (useful for backfilling historical data)
- **Interpolation**: Creates smooth transitions between known values (linear, polynomial)

The choice should reflect the underlying physical or business process generating the data.

In [ ]:
print('Missing values per column:\n', missing)

In [ ]:
# GUIDED: choose a missing value strategy and apply it
# TODO: Replace 'method' / 'limit' as needed or implement interpolation

filled = df.copy()

# Example 1: linear-based interpolation
# filled = filled.interpolate(method='linear')

# Example 2: forward/backward fill
filled = filled.ffill().bfill()

In [ ]:
# EXERCISE: Compare filled vs non-filled data in subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Plot original resampled data (with missing values)
df.iloc[:1000, 0].plot(ax=ax1, title='Original resampled data (with missing values)', 
                               xlabel='Timestamp', ylabel='Value (MW)', color='blue')
ax1.grid(True, alpha=0.3)

# Plot filled data
filled.iloc[:1000, 0].plot(ax=ax2, title='After missing value handling', 
                           xlabel='Timestamp', ylabel='Value (MW)', color='orange')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# GUIDED: Check now many missing values remain
print('Missing values per column:\n', filled.isna().sum())

## 3. Resampling Time Series Data

**Goal:** Transform irregular or high-frequency data into a consistent, regular time grid.

### Why Resampling Matters:
- Creates uniform time steps for modeling algorithms
- Reduces noise through aggregation
- Aligns data with business/analysis timeframes (hourly, daily, weekly)

### Key Considerations:
1. Detect the original (native) sampling frequency
2. Choose an appropriate target frequency (hourly, daily, etc.)
3. Select the right aggregation method:
   - **Mean**: For measurements representing instantaneous values
   - **Sum**: For cumulative values that should be added (e.g., energy production)
   - **Last/First**: For snapshots or state-based values

In [ ]:
# GUIDED: Resampling

TARGET_FREQ = 'D'  # e.g., 'H' hourly, 'D' daily, 'M' monthly
AGG = 'mean'       # domain-dependent; 'sum' might be better for energy if values are power vs. energy

if AGG == 'mean':
    resampled = filled.resample(TARGET_FREQ).mean()
elif AGG == 'sum':
    resampled = filled.resample(TARGET_FREQ).sum()
elif AGG == 'last':
    resampled = filled.resample(TARGET_FREQ).last()
else:
    raise ValueError('Unsupported aggregation')

resampled.head(3)

In [ ]:
# GUIDED: visualize resampled series
plt.figure(figsize=(10, 3))
resampled.iloc[:1000, 0].plot(title=f'Resampled series at {TARGET_FREQ}')
plt.tight_layout()
plt.show()

<div style="background-color: #ffedcc; border-left: 6px solid #ff7518; padding: 10px; margin-bottom: 10px;">
<h2>📝 Exercise: Data Preparation</h2>

Complete the following tasks with the `electricity.parquet` dataset:
1. Load and visualize the raw data
2. Identify and handle missing values 
   - Try different filling methods (e.g., ffill, bfill, interpolate)

3. Resample the data to a suitable frequency
   - Try different resampling frequencies (e.g., hourly vs. daily)


This exercise will help reinforce the concepts of data cleaning and preparation for time series.
</div>

In [ ]:
# Load exercise data

df_excercise = pd.read_parquet("electricity.parquet")
df_excercise['timestamp'] = ...



In [ ]:
# GUIDED: visualize raw series: try differrent households "MT_001", "MT_002", ..., 


In [ ]:
# Check missing values


In [ ]:
# handle missing values

filled_exercise = ...

In [ ]:
# Resampling: try frequency "D", "h", "W" and plot the resampled series

resampled_exercise = ...


## 4. Feature Scaling for Time Series

Proper scaling is essential for many machine learning algorithms to work effectively with time series data.

### Common Scaling Techniques:
- **Z-score (StandardScaler)**: `(x - mean) / std` - Centers data around zero with unit variance
- **Min-Max (MinMaxScaler)**: Scales data to a fixed range [0, 1]
- **Robust Scaler**: Uses median and IQR, making it resistant to outliers

### ⚠️ Important: Avoid Data Leakage
Always fit scalers **only on training data** to prevent information leakage from the validation and test sets. The code below demonstrates the effect of scaling but should not be applied to the entire dataset in a real forecasting scenario.

In [ ]:
# GUIDED: Experiment with different scaling methods
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
# Uncomment ONE of the following scaling methods to see how they affect the data:

# Option 1: Z-Score (Standard) Scaling - normalizes to mean=0, std=1
# rescaled = pd.DataFrame(StandardScaler().fit_transform(resampled.values), 
#                                index=resampled.index, columns=resampled.columns)

# Option 2: MinMax Scaling - scales to range [0, 1]
# rescaled = pd.DataFrame(MinMaxScaler().fit_transform(resampled.values), 
#                                index=resampled.index, columns=resampled.columns)

# Option 3: Robust Scaling - uses median and IQR, less sensitive to outliers
rescaled = pd.DataFrame(RobustScaler().fit_transform(resampled.values), 
                               index=resampled.index, columns=resampled.columns)


In [ ]:
# Visualize the effects of scaling
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Original data
resampled.iloc[:100, 0].plot(ax=ax1, title='Original Data (first 100 days)', 
                            xlabel='Date', ylabel='Value (MW)', color='blue')
ax1.grid(True, alpha=0.3)

# Scaled data
rescaled.iloc[:100, 0].plot(ax=ax2, title='Scaled Data (first 100 days)', 
                              xlabel='Date', ylabel='Scaled Value', color='orange')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# GUIDED: Define functions for training/validation/test scaling
def scale_using_zscore(train_df, val_df, test_df):
    """
    Apply Z-Score (Standard) scaling: (x - mean) / std
    Returns: train_scaled, val_scaled, test_scaled, scaler
    """
    scaler = StandardScaler()
    scaler.fit(train_df.values)
    
    train_scaled = pd.DataFrame(scaler.transform(train_df.values), 
                               index=train_df.index, columns=train_df.columns)
    val_scaled = pd.DataFrame(scaler.transform(val_df.values), 
                             index=val_df.index, columns=val_df.columns)
    test_scaled = pd.DataFrame(scaler.transform(test_df.values), 
                              index=test_df.index, columns=test_df.columns)
    
    return train_scaled, val_scaled, test_scaled, scaler

def scale_using_minmax(train_df, val_df, test_df):
    """
    Apply MinMax scaling: (x - min) / (max - min) -> [0, 1]
    Returns: train_scaled, val_scaled, test_scaled, scaler
    """
    scaler = MinMaxScaler()
    scaler.fit(train_df.values)
    
    train_scaled = pd.DataFrame(scaler.transform(train_df.values), 
                               index=train_df.index, columns=train_df.columns)
    val_scaled = pd.DataFrame(scaler.transform(val_df.values), 
                             index=val_df.index, columns=val_df.columns)
    test_scaled = pd.DataFrame(scaler.transform(test_df.values), 
                              index=test_df.index, columns=test_df.columns)
    
    return train_scaled, val_scaled, test_scaled, scaler

def scale_using_robust(train_df, val_df, test_df):
    """
    Apply Robust scaling: (x - median) / IQR
    Less sensitive to outliers than StandardScaler
    Returns: train_scaled, val_scaled, test_scaled, scaler
    """
    scaler = RobustScaler()
    scaler.fit(train_df.values)
    
    train_scaled = pd.DataFrame(scaler.transform(train_df.values), 
                               index=train_df.index, columns=train_df.columns)
    val_scaled = pd.DataFrame(scaler.transform(val_df.values), 
                             index=val_df.index, columns=val_df.columns)
    test_scaled = pd.DataFrame(scaler.transform(test_df.values), 
                              index=test_df.index, columns=test_df.columns)
    
    return train_scaled, val_scaled, test_scaled, scaler

def print_scaling_stats(train_scaled, val_scaled, test_scaled, method_name):
    """Print statistics for scaled datasets"""
    print(f"\n{method_name} Scaling Statistics:")
    print(f"Train - Mean: {train_scaled.mean().values[0]:.6f}, Std: {train_scaled.std().values[0]:.6f}")
    print(f"Val   - Mean: {val_scaled.mean().values[0]:.6f}, Std: {val_scaled.std().values[0]:.6f}")
    print(f"Test  - Mean: {test_scaled.mean().values[0]:.6f}, Std: {test_scaled.std().values[0]:.6f}")

## 5. Time-Aware Data Splitting

Unlike traditional random splits used in other ML tasks, time series data requires **chronological splitting** to preserve temporal dependencies and avoid look-ahead bias.

### Common Splitting Strategies:
- **Hold-Out Split**: Simple chronological division (e.g., 70% train, 15% validation, 15% test)
- **Rolling Window**: Multiple train/test splits that move forward in time
- **Expanding Window**: Growing training set with a rolling evaluation window

For this introductory block, we'll implement a basic Hold-Out split to establish the foundation for more advanced techniques.

In [ ]:
# GUIDED: chronological split

len_train = int(len(resampled) * 0.7)
len_val = int(len(resampled) * 0.15)

train = resampled.iloc[:len_train]
val = resampled.iloc[len_train:len_train+len_val]
test = resampled.iloc[len_train+len_val:]

print('Split sizes:', len(train), len(val), len(test))
print('Ranges:')
print('Train:', train.index.min(), '→', train.index.max())
print('Val  :', val.index.min(),   '→', val.index.max())
print('Test :', test.index.min(),  '→', test.index.max())

In [ ]:
# GUIDED: Choose and apply scaling method
# Uncomment ONE of the following scaling methods:

# Option 1: Z-Score (Standard) Scaling
train_s, val_s, test_s, SCALER = scale_using_zscore(train, val, test)

# Option 2: MinMax Scaling
# train_s, val_s, test_s, SCALER = scale_using_minmax(train, val, test)
# print_scaling_stats(train_s, val_s, test_s, "MinMax")

# Option 3: Robust Scaling
# train_s, val_s, test_s, SCALER = scale_using_robust(train, val, test)
# print_scaling_stats(train_s, val_s, test_s, "Robust")

train_s.head(3)

In [ ]:
# Print scaling statistics
print_scaling_stats(train_s, val_s, test_s, "Z-Score")

# Visualize original vs scaled data in subplots
fig, ((ax1, ax2)) = plt.subplots(1, 2, figsize=(15, 5))

# Original train data
train.iloc[:1000, 0].plot(ax=ax1, title='Original Train Data (first ~1000 points)', 
                          xlabel='Timestamp', ylabel='Value (MW)', color='blue')
ax1.grid(True, alpha=0.3)

# Scaled train data
train_s.iloc[:1000, 0].plot(ax=ax2, title='Scaled Train Data (first ~1000 points)', 
                            xlabel='Timestamp', ylabel='Scaled Value', color='orange')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<div style="background-color: #ffedcc; border-left: 6px solid #ff7518; padding: 10px; margin-bottom: 10px;">
<h2>📝 Exercise: Train-Validation-Test Split & Scaling</h2>

Apply what you've learned to the electricity dataset:
1. Split the resampled data into train, validation, and test sets using the Hold-Out Split method
2. Apply the most appropriate scaling method 
   - Try different scaling frequencies (e.g., zscore, MinMax)
   - Visualize random samples
3. Verify that scaling was properly applied without data leakage

This exercise will help you practice implementing a proper time series preprocessing pipeline.
</div>

In [ ]:
# split the resampled electricity data in train validation test using the Hold-Out Split method
# 70% train, 15% val, 15% test


train_ex = ...
val_ex = ...
test_ex = ...


In [ ]:
# - rescale the data using the most appropriate method



## 6. Creating Sliding Windows for Sequence Modeling

For supervised learning with time series, we need to transform our sequential data into input-output pairs using a sliding window approach.

### Key Parameters:
- **Window Size**: Number of past time steps to use as input features (e.g., 24 hours of history)
- **Forecast Horizon**: Number of future time steps to predict (e.g., next 6 hours)
- **Stride**: Step size between consecutive windows (smaller values create more overlapping samples)

This technique converts our time series into a format suitable for various modeling approaches, from traditional ML to deep learning architectures like RNNs, LSTMs, and Transformers.

In [ ]:
# GUIDED: windowing utility
def make_windows(series, window_size, forecast_horizon, stride):
    """
    Convert a univariate or multivariate time series into (X, y) windows.
    X shape: (num_windows, window_size, num_features)
    y shape: (num_windows, forecast_horizon) if univariate target
    """
    values = series.values
    Xs, ys = [], []
    
    for i in range(0, len(values) - window_size - forecast_horizon + 1, stride):
        X_window = values[i:i + window_size]
        y_window = values[i + window_size:i + window_size + forecast_horizon]
        Xs.append(X_window)
        ys.append(y_window)
    
    return np.array(Xs), np.array(ys)

WINDOW_SIZE = 24   # past 24 hours
HORIZON = 6       # predict next 6 hours
STRIDE = 1

X_train, y_train = make_windows(train_s, WINDOW_SIZE, HORIZON, STRIDE)
X_val, y_val = make_windows(val_s, WINDOW_SIZE, HORIZON, STRIDE)
X_test, y_test = make_windows(test_s, WINDOW_SIZE, HORIZON, STRIDE)


In [ ]:
# Visualize the shapes and a sample of the windowed data
print(f"Window shapes:")
print(f"X_train: {X_train.shape} (samples, window_size, features)")
print(f"y_train: {y_train.shape} (samples, forecast_horizon)")
print(f"X_val: {X_val.shape}")
print(f"y_val: {y_val.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")

# Plot a few windowed samples to verify correctness
fig, axes = plt.subplots(3, 1, figsize=(10, 8 ))

# Plot first training window
axes[0].plot(range(WINDOW_SIZE), X_train[0, :, 0], 'b-', label='Input window', linewidth=2)
axes[0].plot(range(WINDOW_SIZE, WINDOW_SIZE + HORIZON), y_train[0, :, 0], 'r-', label='Target', linewidth=2, marker='o')
axes[0].set_title('Training Sample 1')
axes[0].set_xlabel('Time steps')
axes[0].set_ylabel('Scaled value')
axes[0].legend()
axes[0].grid(True, alpha=0.3)


# Plot validation window
axes[1].plot(range(WINDOW_SIZE), X_val[0, :, 0], 'b-', label='Input window', linewidth=2)
axes[1].plot(range(WINDOW_SIZE, WINDOW_SIZE + HORIZON), y_val[0, :, 0], 'r-', label='Target', linewidth=2, marker='o')
axes[1].set_title('Validation Sample 1')
axes[1].set_xlabel('Time steps')
axes[1].set_ylabel('Scaled value')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot test window
axes[2].plot(range(WINDOW_SIZE), X_test[0, :, 0], 'b-', label='Input window', linewidth=2)
axes[2].plot(range(WINDOW_SIZE, WINDOW_SIZE + HORIZON), y_test[0, :, 0], 'r-', label='Target', linewidth=2, marker='o')
axes[2].set_title('Test Sample 1')
axes[2].set_xlabel('Time steps')
axes[2].set_ylabel('Scaled value')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<div style="background-color: #ffedcc; border-left: 6px solid #ff7518; padding: 10px; margin-bottom: 10px;">
<h2>📝 Exercise: Creating Sliding Windows</h2>

Apply the windowing technique to the preprocessed electricity dataset:
1. Create input-output window pairs from the train, validation, and test sets
2. Experiment with different window sizes and forecast horizons
   - Test different window sizes (e.g., 12, 24, 48)
   - Adjust the forecast horizon (e.g., 1, 3, 6)

3. Visualize sample windows to ensure they're constructed correctly

This exercise will help you understand how different window parameters affect your model inputs and prediction targets.
</div>

In [ ]:
# create windows for the exercise dataset and plot some samples to verify correctness

WINDOW_SIZE = ...
HORIZON = ...
STRIDE = ...


## ✅ Summary

### What You've Accomplished:
- Loaded and performed quality checks on time series data
- Implemented strategies for handling missing values and resampling
- Applied proper scaling techniques while avoiding data leakage
- Created time-aware train/validation/test splits
- Generated sliding windows for supervised learning approaches

These preprocessing steps are crucial for achieving good performance with any time series forecasting model.